# پروژه پایانی ۱: دیده‌بان نماد (سطح پایه)
**دوره آموزشی صفر تا کوانت - ایستگاه اول: پایتون مقدماتی**

**هدف پروژه:** 
 بررسی وضعیت چند سهم در بازار و تشخیص اینکه کدام‌یک ارزش زیر نظر گرفتن برای خرید و نگهداری دارند


**توسعه‌دهنده:** Mohammad Amin Peymani far

هدف: نوشتن برنامه‌ای که داده‌های واقعی چند سهم را از سایت
tsetmc بگیرد، آن‌ها را تحلیل کند و یک گزارش تمیز تحویل دهد.
----------------------------
1. گرفتن داده: 
 دریافت تاریخچه ۳۰ روزه قیمت و داده‌های حقیقی/حقوقی امروز.

2. تمیزکاری داده (تله‌ها): 
 تبدیل متن به عدد، جدا کردن ریال و تومان، پیدا کردن .کد نماد (insCode) و حل مشکل حروف عربی «ی» و «ک».

3. محاسبات  (فقط با حلقه و لیست، بدون Pandas):
 - تشخیص رنگ کندل (سبز، قرمز، دوجی) و محاسبه درصد بازده روز.
 - محاسبه سقف قیمت ۵ روز اخیر (هفته) و فاصله قیمت فعلی تا آن.
 - محاسبه میانگین حجم ۲۰ روزه و تشخیص حجم غیرعادی (بیشتر از ۲ برابر میانگین).
 - محاسبه ورود/خروج پول حقیقی (خرید حقیقی منهای فروش حقیقی).

4. قانون سیگنال:
 امتیازدهی بر اساس ۴ نشانه و تبدیل آن به کلمات BUY، WATCH یا WAIT.

5. گزارش‌گیری: 
 مرتب‌سازی نتایج از بیشترین امتیاز به کمترین و ذخیره در فایل report.txt با ابزار save_report.

# 1.

In [1]:
import requests
import time

def get_json(url):
    """گرفتن داده از API و تبدیل به دیکشنری/لیست پایتون"""
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0", "Accept": "application/json"},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()

def save_report(lines, filename):
    """ذخیره لیست خطوط در یک فایل متنی"""
    with open(filename, "w", encoding="utf-8") as f:
        for line in lines:
            f.write(line + "\n")
    print(f"گزارش با موفقیت در فایل {filename} ذخیره شد.")

print("ابزارها آماده شدند.")

ابزارها آماده شدند.


# 2. پیدا کردن کد نمادها

In [2]:
# لیست ۱۰ نماد برای بررسی (بخش امتیازی)
symbols_list = ["فولاد", "فملی", "وبملت", "شپنا", "خساپا", "فارس", "شتران", "وبصادر", "کگل", "ومعادن"]
codes = {}

print("در حال دریافت کد نمادها از tsetmc...")
for symbol in symbols_list:
    url = f"https://cdn.tsetmc.com/api/Instrument/GetInstrumentSearch/{symbol}"
    try:
        data = get_json(url)
        if data and "instrumentSearch" in data and len(data["instrumentSearch"]) > 0:
            for item in data["instrumentSearch"]:
                # رفع تله حروف عربی (ی و ک)
                name = item["lVal18AFC"].replace("ي", "ی").replace("ك", "ک").strip()
                if name == symbol:
                    codes[symbol] = item["insCode"]
                    break
    except Exception as e:
        print(f"خطا در جستجوی {symbol}: {e}")
    time.sleep(1) # رعایت ادب API

print(f"تعداد {len(codes)} نماد پیدا شد.")

در حال دریافت کد نمادها از tsetmc...
تعداد 10 نماد پیدا شد.


# 3. محاسبات اصلی (کندل، بازده، سقف هفته، حجم، پول حقیقی)

In [3]:
results = [] 

print("\nدر حال دریافت و پردازش داده‌های امروز...")
for symbol, code in codes.items():
    try:
        # دریافت تاریخچه ۳۰ روزه و داده‌های حقیقی امروز
        days = get_json(f"https://cdn.tsetmc.com/api/ClosingPrice/GetClosingPriceDailyList/{code}/30")["closingPriceDaily"]
        client_data = get_json(f"https://cdn.tsetmc.com/api/ClientType/GetClientTypeHistory/{code}")["clientType"][0]
        
        d = days[0] # تازه‌ترین روز
        last_close = int(d["pClosing"])
        price_first = int(d["priceFirst"])
        price_yesterday = int(d["priceYesterday"])
        
        # ۱. کندل و بازده
        body = last_close - price_first
        color = "GREEN" if body > 0 else ("RED" if body < 0 else "DOJI")
        change = ((last_close - price_yesterday) / price_yesterday) * 100
        
        # ۲. سقف هفته و فاصله
        week = days[0:5]
        week_high = week[0]["priceMax"]
        for w in week:
            if w["priceMax"] > week_high:
                week_high = w["priceMax"]
        gap_to_high = ((week_high - last_close) / last_close) * 100
        
        # ۳. حجم غیرعادی (میانگین ۲۰ روزه)
        total_vol = sum([w["qTotTran5J"] for w in days[1:21]])
        avg_vol = total_vol / 20
        today_vol = d["qTotTran5J"]
        vol_alert = today_vol > (2 * avg_vol)
        
        # ۴. پول حقیقی
        money_in = (client_data["buy_I_Value"] - client_data["sell_I_Value"]) // 10 # تومان
        
        # ماموریت امتیازی: تشخیص صف خرید
        is_queue = (d["priceMax"] == d["priceMin"] == d["pClosing"]) and (change > 0)

        results.append({
            "symbol": symbol, "close": last_close, "change": change,
            "candle": color, "gap_to_high": gap_to_high, "vol_alert": vol_alert,
            "money_in": money_in, "date": d["dEven"], "is_queue": is_queue
        })
        
    except Exception as e:
        print(f"خطا در پردازش {symbol}: {e}")
    time.sleep(1)

print("پردازش داده‌ها به پایان رسید.")


در حال دریافت و پردازش داده‌های امروز...
پردازش داده‌ها به پایان رسید.


# ۴. قانون سیگنال (امتیازدهی و تشخیص خرید/فروش)

In [4]:
for r in results:
    score = 0
    
    # ۴ نشانه اصلی
    if r["candle"] == "GREEN": score += 1
    if r["money_in"] > 0: score += 1
    if r["vol_alert"] and r["candle"] == "GREEN": score += 1
    if r["gap_to_high"] < 2: score -= 1
    
    # تعیین سیگنال (با اولویت صف خرید)
    if r["is_queue"]:
        signal = "QUEUE" # ماموریت امتیازی
    elif score >= 2:
        signal = "BUY"
    elif score == 1:
        signal = "WATCH"
    else:
        signal = "WAIT"
        
    r["score"] = score
    r["signal"] = signal

print("امتیازدهی و سیگنال‌دهی انجام شد.")

امتیازدهی و سیگنال‌دهی انجام شد.


# ۵. مرتب‌سازی و تولید گزارش 

In [5]:
# مرتب‌سازی نتایج از بیشترین امتیاز به کمترین (با لیست تاپل)
pairs = [(-r["score"], r["symbol"]) for r in results]
pairs = sorted(pairs)

# ساخت خطوط گزارش
report_lines = []
report_lines.append(f"--- دیده‌بان نماد - تاریخ: {results[0]['date'] if results else 'نامشخص'} ---")
report_lines.append(f"{'سیگنال':<10} | {'امتیاز':<5} | {'نماد':<10} | {'پایانی(ریال)':<15} | {'تغییر%':<10} | {'کندل':<8} | {'پول حقیقی(تومان)':<15}")
report_lines.append("-" * 95)

# ۱. چاپ وضعیت تمام نمادها (برای اینکه گزارش خالی نباشد)
for neg_score, symbol in pairs:
    for r in results:
        if r["symbol"] == symbol:
            line = f"{r['signal']:<10} | {r['score']:<5} | {r['symbol']:<10} | {r['close']:>13,} | {r['change']:>8.2f}% | {r['candle']:<8} | {r['money_in']:>15,}"
            report_lines.append(line)
            break

# ۲. جداسازی نمادهای خرید (بخش امتیازی)
report_lines.append("\n" + "="*50)
report_lines.append("*** نمادهای دارای سیگنال BUY یا QUEUE ***")
buy_count = 0
for neg_score, symbol in pairs:
    for r in results:
        if r["symbol"] == symbol and r["signal"] in ["BUY", "QUEUE"]:
            report_lines.append(f"{r['signal']} -> {r['symbol']}")
            buy_count += 1
            break

if buy_count == 0:
    report_lines.append("امروز هیچ نمادی سیگنال BUY یا QUEUE نداشت.")

# ذخیره در فایل
filename = "report_17Aban.txt" # نام فایل را هر روز تغییر دهید
save_report(report_lines, filename)

# نمایش در خروجی
for line in report_lines:
    print(line)

گزارش با موفقیت در فایل report_17Aban.txt ذخیره شد.
--- دیده‌بان نماد - تاریخ: 20261109 ---
سیگنال     | امتیاز | نماد       | پایانی(ریال)    | تغییر%     | کندل     | پول حقیقی(تومان)
-----------------------------------------------------------------------------------------------
BUY        | 2     | شتران      |         6,090 |    -0.98% | GREEN    | 7,373,680,443.0
BUY        | 2     | شپنا       |        11,880 |     0.00% | GREEN    | 4,087,656,257.0
BUY        | 2     | فملی       |        28,320 |    -2.38% | GREEN    | 12,575,354,520.0
WATCH      | 1     | خساپا      |           598 |     0.50% | GREEN    | 18,161,809,823.0
WATCH      | 1     | فولاد      |         4,670 |     2.86% | GREEN    | 254,084,500,361.0
WATCH      | 1     | ومعادن     |         4,180 |     2.20% | GREEN    | 75,048,227,680.0
WATCH      | 1     | کگل        |         3,920 |     2.62% | GREEN    | 186,148,675,525.0
QUEUE      | 0     | فارس       |        15,340 |     2.95% | DOJI     | 324,597,224,094

## ۷. ماموریت‌های امتیازی

In [6]:
print("\n--- شروع ماموریت‌های امتیازی (بخش API) ---")

# ماموریت ۱: چاپ کلیدهای گواهی سپرده از tgju
print("\n[ماموریت ۱] کلیدهای ime_certificate:")
try:
    current_tgju = get_json("https://call1.tgju.org/ajax.json")["current"]
    for key in current_tgju:
        if "ime_certificate" in key:
            print(f" - {key}: {current_tgju[key]['p']}")
except:
    print("خطا در دریافت داده tgju")

# ماموریت ۲: میانگین قیمت تتر از ۴ صرافی
print("\n[ماموریت ۲] میانگین قیمت تتر:")
tether_prices = []
try: tether_prices.append(int(get_json("https://apiv2.nobitex.ir/market/stats?srcCurrency=usdt&dstCurrency=rls")["stats"]["usdt-rls"]["latest"]) // 10)
except: pass
try:
    markets = get_json("https://api.wallex.ir/hector/web/v1/markets?version=2")["result"]["markets"]
    for m in markets:
        if m["symbol"] == "USDTTMN": tether_prices.append(int(float(m["price"]))); break
except: pass
try:
    rows = get_json("https://api.bitpin.ir/v4/mkt/prices/")
    for row in rows:
        if row["code"] == "USDT_IRT": tether_prices.append(int(float(row["price"]))); break
except: pass
try: tether_prices.append(get_json("https://service.tetherland.com/api/v4/currencies")["price"])
except: pass

if tether_prices:
    print(f"میانگین تتر: {sum(tether_prices)/len(tether_prices):,.0f} تومان")

# ماموریت ۳: انحراف معیار سکه بهار با حلقه
print("\n[ماموریت ۳] انحراف معیار ۳۰ روزه سکه بهار:")
try:
    rows = get_json("https://api.tgju.org/v1/market/indicator/summary-table-data/sekeb?length=30")["data"]
    prices = [int(row[3].replace(",", "")) for row in rows]
    returns = [((prices[i] - prices[i+1]) / prices[i+1]) * 100 for i in range(len(prices)-1)]
    mean = sum(returns) / len(returns)
    variance = sum([(r - mean) ** 2 for r in returns]) / (len(returns) - 1)
    print(f"انحراف معیار: {variance ** 0.5:.2f}%")
except:
    print("خطا در محاسبه انحراف معیار")


--- شروع ماموریت‌های امتیازی (بخش API) ---

[ماموریت ۱] کلیدهای ime_certificate:
 - ime_certificate_silver_bars: 4,500,000
 - ime_certificate_gold_bars: 33,635,430
 - ime_certificate_coin: 2,565,950,000

[ماموریت ۲] میانگین قیمت تتر:
میانگین تتر: 264,675 تومان

[ماموریت ۳] انحراف معیار ۳۰ روزه سکه بهار:
انحراف معیار: 1.49%


### 7. توضیح قانون سیگنال:

### توضیح قانون سیگنال و انتخاب اعداد

**قانون سیگنال من:**
- کندل سبز (بسته شدن بالاتر از بازگشایی): **+۱ امتیاز**
- ورود پول حقیقی مثبت (خریداران قوی‌تر از فروشندگان): **+۱ امتیاز**
- حجم غیرعادی (بیش از ۲ برابر میانگین ۲۰ روزه) همراه با کندل سبز: **+۱ امتیاز**
- فاصله کمتر از ۲ درصد تا سقف هفته (نزدیک به مقاومت): **-۱ امتیاز**

**دلیل انتخاب اعداد:**
- **حجم غیرعادی ۲ برابر:** در تحلیل تکنیکال، حجمی که دو برابر میانگین باشد نشانه ورود پول هوشمند یا یک اتفاق مهم است. اگر این حجم با کندل سبز همراه شود، نشان‌دهنده قدرت خریداران است.
- **فاصله ۲ درصد تا سقف هفته:** اگر قیمت خیلی به سقف ۵ روز اخیر نزدیک باشد (کمتر از ۲٪)، احتمال برخورد با مقاومت و برگشت قیمت وجود دارد، بنابراین یک امتیاز منفی در نظر گرفته می‌شود.
- **آستانه‌های سیگنال:**
  - **BUY:** امتیاز ۲ یا بیشتر (یعنی حداقل دو نشانه مثبت قوی با هم).
  - **WATCH:** امتیاز ۱ (یک نشانه مثبت).
  - **WAIT:** امتیاز ۰ یا کمتر (هیچ نشانه مثبتی وجود ندارد یا در نزدیکی مقاومت است).
  - **QUEUE:** اگر قیمت سقف، کف و پایانی برابر باشد و بازده مثبت باشد، یعنی صف خرید است و اولویت آن از سایر سیگنال‌ها بالاتر است.

##  ۹. بک‌تست استراتژی روی ۳۰ روز گذشته:

In [7]:
print("\n--- شروع بک‌تست استراتژی روی ۳۰ روز گذشته (فقط برای فولاد) ---")

# برای سادگی، بک‌تست را فقط روی نماد اول (مثلاً فولاد) انجام می‌دهیم
target_symbol = list(codes.keys())[0]
target_code = codes[target_symbol]

# دریافت ۴۰ روز داده (۳۰ روز برای بک‌تست + ۱۰ روز برای محاسبه میانگین حجم ۲۰ روزه)
days_bt = get_json(f"https://cdn.tsetmc.com/api/ClosingPrice/GetClosingPriceDailyList/{target_code}/40")["closingPriceDaily"]

wins = 0
trades = 0

# از ایندکس ۲۰ به بعد شروع می‌کنیم چون به ۲۰ روز داده قبلی برای میانگین حجم نیاز داریم
for i in range(len(days_bt) - 21, 0, -1): 
    t = days_bt[i] # روزی که می‌خواهیم سیگنال آن را بررسی کنیم
    
    # ۱. محاسبه نشانه‌ها برای روز t
    close = int(t["pClosing"])
    first = int(t["priceFirst"])
    color = "GREEN" if close > first else "RED"
    
    # محاسبه میانگین حجم ۲۰ روز قبل از روز t
    total_vol = sum([w["qTotTran5J"] for w in days_bt[i+1 : i+21]])
    avg_vol = total_vol / 20
    vol_alert = t["qTotTran5J"] > (2 * avg_vol)
    
    # فرض می‌کنیم پول حقیقی مثبت است (برای سادگی در بک‌تست، چون تاریخچه حقیقی را نگرفتیم)
    money_in_positive = True 
    
    # ۲. قانون سیگنال (بدون فاصله تا سقف برای سادگی)
    score = 0
    if color == "GREEN": score += 1
    if money_in_positive: score += 1
    if vol_alert and color == "GREEN": score += 1
    
    signal = "BUY" if score >= 2 else "WATCH"
    
    # ۳. بررسی نتیجه (فردای آن روز چه شد؟)
    # چون داده‌ها از جدید به قدیم هستند، روز بعد از t، در ایندکس i-1 قرار دارد
    if signal == "BUY":
        trades += 1
        next_day_close = int(days_bt[i-1]["pClosing"])
        # اگر قیمت پایانی فردا از قیمت پایانی امروز بیشتر بود، یعنی معامله برنده بوده
        if next_day_close > close:
            wins += 1

if trades > 0:
    win_rate = (wins / trades) * 100
    print(f"نماد: {target_symbol}")
    print(f"تعداد سیگنال‌های BUY در ۳۰ روز گذشته: {trades}")
    print(f"تعداد معاملات برنده: {wins}")
    print(f"درصد موفقیت (Win Rate): {win_rate:.2f}%")
else:
    print("در ۳۰ روز گذشته هیچ سیگنال BUY صادر نشده است.")


--- شروع بک‌تست استراتژی روی ۳۰ روز گذشته (فقط برای فولاد) ---
نماد: فولاد
تعداد سیگنال‌های BUY در ۳۰ روز گذشته: 7
تعداد معاملات برنده: 6
درصد موفقیت (Win Rate): 85.71%


## 9. توضیح بک‌تست و مدیریت ریسک:

### نتایج بک‌تست و مدیریت ریسک

در بخش بک‌تست، قانون سیگنال خود را روی ۳۰ روز گذشته یک نماد شاخص (مانند فولاد) آزمایش کردیم.
- **نتیجه بک‌تست:** این آزمایش نشان می‌دهد که قانون ما در گذشته چقدر موفق بوده است. (اگر درصد موفقیت بالای ۵۰٪ باشد، یعنی استراتژی برتری نسبت به شانس دارد).
- **مدیریت ریسک:** در دنیای واقعی، حتی بهترین استراتژی‌ها هم ضرر می‌دهند. یک معامله‌گر حرفه‌ای هرگز بیشتر از ۱ تا ۲ درصد کل سرمایه خود را در یک معامله (مثلاً خرید یک سهم) ریسک نمی‌کند. یعنی اگر قیمت به ضرر رفت (حد ضرر)، فقط ۱ درصد سرمایه از دست می‌رود، نه کل آن.